In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, time
import numpy as np
import pandas as pd
import requests, getpass
from difflib import SequenceMatcher

FILE_PATH="/content/drive/MyDrive/bananika.csv"
OUT_DIR="/content/drive/MyDrive/openalex_enrich_out"
MIN_YEAR=2012
TITLE_COL="title"
YEAR_COL="year"

os.makedirs(OUT_DIR, exist_ok=True)

df=pd.read_csv(FILE_PATH, engine="python", encoding="latin-1", on_bad_lines="skip")
df=df.loc[:, ~df.columns.str.match(r"^Unnamed")].copy()
df[YEAR_COL]=pd.to_numeric(df[YEAR_COL], errors="coerce")
df=df.dropna(subset=[YEAR_COL, TITLE_COL]).copy()
df[YEAR_COL]=df[YEAR_COL].astype(int)
df2012=df[df[YEAR_COL]>=MIN_YEAR].copy()

Mounted at /content/drive


In [2]:
PAPERS_PATH=f"{OUT_DIR}/nips-papers_enriched_openalex.csv"
AUTHORS_PATH=f"{OUT_DIR}/nips-authors_unique_openalex.csv"

papers_enriched_df=pd.read_csv(PAPERS_PATH) if os.path.exists(PAPERS_PATH) else pd.DataFrame()
authors_df=pd.read_csv(AUTHORS_PATH) if os.path.exists(AUTHORS_PATH) else pd.DataFrame()

authors_map={}
if len(authors_df) and "oa_author_id" in authors_df.columns and "year" in authors_df.columns:
    for _,r in authors_df.iterrows():
        aid=r.get("oa_author_id")
        yr=r.get("year")
        if not isinstance(aid,str) or not aid:
            continue
        try:
            yr=int(yr)
        except:
            continue
        try:
            inst=json.loads(r.get("institutions_json","[]") or "[]")
        except:
            inst=[]
        try:
            raw=json.loads(r.get("raw_affiliation_strings_json","[]") or "[]")
        except:
            raw=[]
        authors_map[(aid, yr)]={
            "oa_author_id": aid,
            "year": yr,
            "author_name": r.get("author_name"),
            "orcid": r.get("orcid") if isinstance(r.get("orcid"),str) else None,
            "institutions": set((x.get("id"), x.get("name")) for x in inst if isinstance(x,dict)),
            "raw_affils": set(str(x) for x in raw if x is not None),
        }

In [3]:
OPENALEX_API_KEY=getpass.getpass("OpenAlex API key: ").strip()
assert OPENALEX_API_KEY

s=requests.Session()

def ct(x):
    x=(x or "").strip()
    return re.sub(r"\s+"," ",x)

def qf(x):
    x=(x or "")
    x=x.replace("\\","\\\\").replace('"','\\"')
    return f'"{x}"'

def tsim(a,b):
    a=re.sub(r"\s+"," ",(a or "").strip().lower())
    b=re.sub(r"\s+"," ",(b or "").strip().lower())
    return SequenceMatcher(None,a,b).ratio()

def norm_title(t):
    t=ct(t)
    t=re.sub(r"\$.*?\$"," ",t)
    t=re.sub(r"\\[a-zA-Z]+"," ",t)
    t=re.sub(r"[{}]"," ",t)
    t=re.sub(r"\s+"," ",t).strip()
    return t

def oa_lookup(title, year, k=25, timeout=30, sim_min=0.92, year_max_delta=2):
    title=norm_title(title)
    if not title: return None, {"status":"skip_no_title"}

    def req(filt):
        r=s.get("https://api.openalex.org/works", params={
            "filter": filt,
            "per_page": int(k),
            "select": "id,display_name,publication_year,doi,cited_by_count,primary_topic,topics,authorships",
            "api_key": OPENALEX_API_KEY,
        }, timeout=timeout)
        info={"http_status":r.status_code,"remaining_usd":r.headers.get("X-RateLimit-Remaining-USD"),
              "cost_usd":r.headers.get("X-RateLimit-Cost-USD"),"reset_sec":r.headers.get("X-RateLimit-Reset"),
              "retry_after":r.headers.get("Retry-After")}
        if r.status_code==429: return None, {"status":"stop_429_budget_or_rate", **info}
        if r.status_code!=200: return None, {"status":f"error_http_{r.status_code}", **info}
        res=(r.json().get("results") or [])
        return res, {"status":"ok", **info}

    y=f"{year-1}|{year}|{year+1}"
    res,info=req(f"title.search:{qf(title)},publication_year:{y}")
    if info.get("status")!="ok": return None, info
    if res:
        best=max(res, key=lambda w: tsim(title, norm_title(w.get("display_name"))))
        return best, {"status":"ok", "best_title_sim": tsim(title, norm_title(best.get("display_name"))), **info}

    res,info=req(f"title.search:{qf(title)}")
    if info.get("status")!="ok": return None, info
    if not res: return None, {"status":"no_match", **info}

    best=max(res, key=lambda w: tsim(title, norm_title(w.get("display_name"))))
    bs=tsim(title, norm_title(best.get("display_name")))
    by=best.get("publication_year")
    if bs < sim_min: return None, {"status":"no_match_low_sim", "best_title_sim": bs, **info}
    if by is not None and abs(int(by)-int(year)) > year_max_delta:
        return None, {"status":"no_match_year_far", "best_title_sim": bs, "best_year": by, **info}

    return best, {"status":"ok_fallback", "best_title_sim": bs, "best_year": by, **info}

OpenAlex API key: ··········


In [4]:
done=set()
if len(papers_enriched_df) and "src_index" in papers_enriched_df.columns and "oa_status" in papers_enriched_df.columns:
    done=set(papers_enriched_df.loc[papers_enriched_df["oa_status"].isin(["ok","ok_fallback"]),
                                "src_index"].dropna().astype(int).tolist())

todo=df2012[~df2012.index.isin(done)].copy()
print("df2012:", df2012.shape, "done_ok:", len(done), "todo:", todo.shape)

new=[]
t0=time.time()

for i,(_,row) in enumerate(todo.iterrows(),1):
    year=int(row[YEAR_COL]); title=str(row[TITLE_COL])
    work,info=oa_lookup(title, year, k=5)
    if info.get("status")=="stop_429_budget_or_rate":
        print("STOP 429", {k:info.get(k) for k in ["remaining_usd","cost_usd","reset_sec","retry_after"]})
        break

    rec={
        "src_index": int(row.name),
        "year": year,
        "title": ct(title),
        "oa_status": info.get("status"),
        "oa_http_status": info.get("http_status"),
        "oa_title_sim": info.get("best_title_sim"),
        "oa_remaining_usd": info.get("remaining_usd"),
        "oa_cost_usd": info.get("cost_usd"),
        "oa_reset_sec": info.get("reset_sec"),
    }
    for col in ["hash_id","pdf_url"]:
        if col in row:
            rec[col]=row[col]

    if work is not None:
        pt=work.get("primary_topic") or {}
        rec.update({
            "oa_work_id": work.get("id"),
            "oa_display_name": work.get("display_name"),
            "oa_doi_url": work.get("doi"),
            "oa_doi": (work.get("doi") or "").replace("https://doi.org/",""),
            "oa_cited_by_count": work.get("cited_by_count"),
            "oa_primary_topic": pt.get("display_name"),
            "oa_primary_topic_score": pt.get("score"),
            "oa_domain": (pt.get("domain") or {}).get("display_name") if isinstance(pt.get("domain"),dict) else None,
            "oa_field": (pt.get("field") or {}).get("display_name") if isinstance(pt.get("field"),dict) else None,
            "oa_subfield": (pt.get("subfield") or {}).get("display_name") if isinstance(pt.get("subfield"),dict) else None,
        })
        topics=[{
            "id": t.get("id"),
            "name": t.get("display_name"),
            "score": t.get("score"),
            "domain": (t.get("domain") or {}).get("display_name") if isinstance(t.get("domain"),dict) else None,
            "field": (t.get("field") or {}).get("display_name") if isinstance(t.get("field"),dict) else None,
            "subfield": (t.get("subfield") or {}).get("display_name") if isinstance(t.get("subfield"),dict) else None,
        } for t in (work.get("topics") or [])[:5]]
        rec["oa_topics_top5_json"]=json.dumps(topics, ensure_ascii=False)

        auths=work.get("authorships") or []
        ids=[]; names=[]; pos=[]
        for a in auths:
            au=a.get("author") or {}
            aid=au.get("id")
            if not aid:
                continue
            ids.append(aid); names.append(au.get("display_name")); pos.append(a.get("author_position"))

            key=(aid, year)
            e=authors_map.get(key)
            if e is None:
                e={
                    "oa_author_id": aid,
                    "year": year,
                    "author_name": au.get("display_name"),
                    "orcid": au.get("orcid"),
                    "institutions": set(),
                    "raw_affils": set(),
                }
                authors_map[key]=e
            else:
                if not e.get("author_name") and au.get("display_name"):
                    e["author_name"]=au.get("display_name")
                if not e.get("orcid") and au.get("orcid"):
                    e["orcid"]=au.get("orcid")

            for inst in (a.get("institutions") or []):
                e["institutions"].add((inst.get("id"), inst.get("display_name")))
            for s2 in (a.get("raw_affiliation_strings") or []):
                if s2:
                    e["raw_affils"].add(str(s2))

        rec["oa_author_ids_json"]=json.dumps(ids, ensure_ascii=False)
        rec["oa_author_names_json"]=json.dumps(names, ensure_ascii=False)
        rec["oa_author_positions_json"]=json.dumps(pos, ensure_ascii=False)

    new.append(rec)
    if i%50==0:
        print(i, "new:", len(new), "author_year:", len(authors_map), "remaining_usd:", rec.get("oa_remaining_usd"), "elapsed:", round(time.time()-t0,1))

if len(new):
    papers_enriched_df=pd.concat([papers_enriched_df, pd.DataFrame(new)], ignore_index=True)

if len(papers_enriched_df) and "src_index" in papers_enriched_df.columns:
    papers_enriched_df["_ok"]=(papers_enriched_df.get("oa_status")=="ok").astype(int)
    papers_enriched_df=papers_enriched_df.sort_values(["src_index","_ok"], ascending=[True,False]).drop_duplicates(["src_index"], keep="first").drop(columns=["_ok"])

authors_rows=[]
for (aid, yr), v in authors_map.items():
    inst_list=sorted(list(v["institutions"]), key=lambda x: (x[1] or "", x[0] or ""))
    authors_rows.append({
        "oa_author_id": v["oa_author_id"],
        "year": int(v["year"]),
        "author_name": v.get("author_name"),
        "orcid": v.get("orcid"),
        "institutions_json": json.dumps([{"id":iid,"name":iname} for iid,iname in inst_list], ensure_ascii=False),
        "raw_affiliation_strings_json": json.dumps(sorted(list(v["raw_affils"])), ensure_ascii=False),
        "n_institutions": len(inst_list),
    })
authors_df=pd.DataFrame(authors_rows)

done=set(papers_enriched_df.loc[papers_enriched_df["oa_status"]=="ok","src_index"].dropna().astype(int).tolist()) if len(papers_enriched_df) else set()
todo=df2012[~df2012.index.isin(done)].copy()

print("papers_enriched_df:", papers_enriched_df.shape)
print("authors_df:", authors_df.shape)
print("todo:", todo.shape)

papers_enriched_df.to_csv(PAPERS_PATH, index=False)
authors_df.to_csv(AUTHORS_PATH, index=False)
print("saved:", PAPERS_PATH)
print("saved:", AUTHORS_PATH)

df2012: (20326, 12) done_ok: 20322 todo: (4, 12)
papers_enriched_df: (20326, 25)
authors_df: (67933, 7)
todo: (4, 12)
saved: /content/drive/MyDrive/openalex_enrich_out/nips-papers_enriched_openalex.csv
saved: /content/drive/MyDrive/openalex_enrich_out/nips-authors_unique_openalex.csv


In [5]:
papers_enriched_df["oa_status"].value_counts()

,count
oa_status,
ok,20322
no_match,2
error_http_400,2


In [6]:
papers_enriched_df.columns

Index(['src_index', 'year', 'title', 'oa_status', 'oa_http_status',
       'oa_title_sim', 'oa_remaining_usd', 'oa_cost_usd', 'oa_reset_sec',
       'hash_id', 'pdf_url', 'oa_work_id', 'oa_display_name', 'oa_doi_url',
       'oa_doi', 'oa_cited_by_count', 'oa_primary_topic',
       'oa_primary_topic_score', 'oa_domain', 'oa_field', 'oa_subfield',
       'oa_topics_top5_json', 'oa_author_ids_json', 'oa_author_names_json',
       'oa_author_positions_json'],
      dtype='object')

In [7]:
authors_df.columns

Index(['oa_author_id', 'year', 'author_name', 'orcid', 'institutions_json',
       'raw_affiliation_strings_json', 'n_institutions'],
      dtype='object')

In [8]:
bad = papers_enriched_df[papers_enriched_df["oa_status"].isin(["no_match","error_http_400"])][["src_index","year","title","oa_status"]]
pd.set_option("display.max_colwidth", None)
bad

,src_index,year,title,oa_status
2715,7166,2017,"Adaptive Accelerated Gradient Converging Method under H\""{o}lderian Error Bound Condition",no_match
7239,11690,2021,"Skyformer: Remodel Self-Attention with Gaussian Kernel and Nystr\""om Method",no_match
11075,15526,2022,"Alleviating ""Posterior Collapse'' in Deep Topic Models via Policy Gradient",error_http_400
16582,21033,2024,"Differential Privacy in Scalable General Kernel Learning via $K$-means Nystr{\""o}m Random Features",error_http_400


In [9]:
# i never said im a good data engineer

import requests, json

titles = [
  "Adaptive Accelerated Gradient Converging Methods under Holderian Error Bound Condition",
  "Skyformer: Remodel Self-Attention with Gaussian Kernel and Nyström Method",
  "Differential Privacy in Scalable General Kernel Learning via $K$-means Nystr{\"o}m Random Features",
]

KEY = OPENALEX_API_KEY
S = requests.Session()

for t in titles:
    r = S.get(
        "https://api.openalex.org/works",
        params={
            "search": t,
            "per_page": 5,
            "select": "id,display_name,publication_year,doi,cited_by_count",
            "api_key": KEY,
        },
        timeout=30,
    )
    print("\nTITLE:", t[:120])
    print("HTTP:", r.status_code)
    if r.status_code != 200:
        print(r.text[:300])
        continue
    for j,w in enumerate((r.json().get("results") or []), 1):
        print(j, w.get("publication_year"), w.get("cited_by_count"), w.get("doi"), w.get("display_name"))


TITLE: Adaptive Accelerated Gradient Converging Methods under Holderian Error Bound Condition
HTTP: 200
1 2016 10 https://doi.org/10.48550/arxiv.1611.07609 Adaptive Accelerated Gradient Converging Methods under Holderian Error Bound Condition
2 2021 1 https://doi.org/10.21538/0134-4889-2021-27-4-175-188 Адаптивные методы градиентного типа для задач оптимизации с относительной точностью и острым минимумом
3 2019 0 https://doi.org/10.17077/etd.a1iu-1h9f Accelerating convex optimization in machine learning by leveraging functional growth conditions

TITLE: Skyformer: Remodel Self-Attention with Gaussian Kernel and Nyström Method
HTTP: 200
1 2021 0 https://doi.org/10.48550/arxiv.2111.00035 Skyformer: Remodel Self-Attention with Gaussian Kernel and Nyström Method

TITLE: Differential Privacy in Scalable General Kernel Learning via $K$-means Nystr{"o}m Random Features
HTTP: 200
1 2024 0 https://doi.org/10.52202/079017-0666 Differential Privacy in Scalable General Kernel Learning via $K$-mea

In [13]:
import json, pandas as pd, requests

if "authors_map" not in globals():
    authors_map = {}
    if "authors_df" in globals() and len(authors_df) and "oa_author_id" in authors_df.columns and "year" in authors_df.columns:
        for _,r in authors_df.iterrows():
            aid=r.get("oa_author_id"); yr=r.get("year")
            if not isinstance(aid,str) or not aid:
                continue
            try: yr=int(yr)
            except:
                continue
            try: inst=json.loads(r.get("institutions_json","[]") or "[]")
            except: inst=[]
            try: raw=json.loads(r.get("raw_affiliation_strings_json","[]") or "[]")
            except: raw=[]
            authors_map[(aid, yr)] = {
                "oa_author_id": aid,
                "year": yr,
                "author_name": r.get("author_name"),
                "orcid": r.get("orcid") if isinstance(r.get("orcid"),str) else None,
                "institutions": set((x.get("id"), x.get("name")) for x in inst if isinstance(x,dict)),
                "raw_affils": set(str(x) for x in raw if x is not None),
            }

if "s" not in globals():
    s = requests.Session()

FIXES = [
    (7166,  "https://doi.org/10.48550/arxiv.1611.07609"),   # Adaptive Accelerated...
    (11690, "https://doi.org/10.48550/arxiv.2111.00035"),   # Skyformer...
    (21033, "https://doi.org/10.52202/079017-0666"),        # Differential Privacy...
]

def fetch_work_by_doi(doi_url):
    r = s.get(
        "https://api.openalex.org/works",
        params={
            "filter": f"doi:{doi_url}",
            "per_page": 1,
            "select": "id,display_name,publication_year,doi,cited_by_count,primary_topic,topics,authorships",
            "api_key": OPENALEX_API_KEY,
        },
        timeout=30,
    )
    if r.status_code != 200:
        return None, {"http_status": r.status_code, "body": r.text[:200]}
    res = (r.json().get("results") or [])
    return (res[0] if res else None), {"http_status": 200}

def overwrite_paper_row(src_index, work):
    m = papers_enriched_df["src_index"].astype(int) == int(src_index)
    if not m.any():
        return False

    year = int(papers_enriched_df.loc[m, "year"].iloc[0])

    pt = work.get("primary_topic") or {}
    topics = [{
        "id": t.get("id"),
        "name": t.get("display_name"),
        "score": t.get("score"),
        "domain": (t.get("domain") or {}).get("display_name") if isinstance(t.get("domain"),dict) else None,
        "field": (t.get("field") or {}).get("display_name") if isinstance(t.get("field"),dict) else None,
        "subfield": (t.get("subfield") or {}).get("display_name") if isinstance(t.get("subfield"),dict) else None,
    } for t in (work.get("topics") or [])[:5]]

    auths = work.get("authorships") or []
    ids=[]; names=[]; pos=[]
    for a in auths:
        au = a.get("author") or {}
        aid = au.get("id")
        if not aid:
            continue
        ids.append(aid)
        names.append(au.get("display_name"))
        pos.append(a.get("author_position"))

        key = (aid, year)
        e = authors_map.get(key)
        if e is None:
            e = {"oa_author_id": aid, "year": year, "author_name": au.get("display_name"), "orcid": au.get("orcid"),
                 "institutions": set(), "raw_affils": set()}
            authors_map[key] = e
        else:
            if not e.get("author_name") and au.get("display_name"): e["author_name"] = au.get("display_name")
            if not e.get("orcid") and au.get("orcid"): e["orcid"] = au.get("orcid")

        for inst in (a.get("institutions") or []):
            e["institutions"].add((inst.get("id"), inst.get("display_name")))
        for s2 in (a.get("raw_affiliation_strings") or []):
            if s2: e["raw_affils"].add(str(s2))

    papers_enriched_df.loc[m, "oa_status"] = "ok_manual_doi"
    papers_enriched_df.loc[m, "oa_http_status"] = 200
    papers_enriched_df.loc[m, "oa_work_id"] = work.get("id")
    papers_enriched_df.loc[m, "oa_display_name"] = work.get("display_name")
    papers_enriched_df.loc[m, "oa_doi_url"] = work.get("doi")
    papers_enriched_df.loc[m, "oa_doi"] = (work.get("doi") or "").replace("https://doi.org/","")
    papers_enriched_df.loc[m, "oa_cited_by_count"] = work.get("cited_by_count")
    papers_enriched_df.loc[m, "oa_primary_topic"] = pt.get("display_name")
    papers_enriched_df.loc[m, "oa_primary_topic_score"] = pt.get("score")
    papers_enriched_df.loc[m, "oa_domain"] = (pt.get("domain") or {}).get("display_name") if isinstance(pt.get("domain"),dict) else None
    papers_enriched_df.loc[m, "oa_field"]  = (pt.get("field")  or {}).get("display_name") if isinstance(pt.get("field"),dict) else None
    papers_enriched_df.loc[m, "oa_subfield"] = (pt.get("subfield") or {}).get("display_name") if isinstance(pt.get("subfield"),dict) else None
    papers_enriched_df.loc[m, "oa_topics_top5_json"] = json.dumps(topics, ensure_ascii=False)
    papers_enriched_df.loc[m, "oa_author_ids_json"] = json.dumps(ids, ensure_ascii=False)
    papers_enriched_df.loc[m, "oa_author_names_json"] = json.dumps(names, ensure_ascii=False)
    papers_enriched_df.loc[m, "oa_author_positions_json"] = json.dumps(pos, ensure_ascii=False)
    return True

fixed = 0
for sid, doi_url in FIXES:
    work, info = fetch_work_by_doi(doi_url)
    if work is None:
        print("FAILED DOI lookup:", sid, doi_url, info)
        continue
    ok = overwrite_paper_row(sid, work)
    print("patched" if ok else "missing src_index", sid, "| OA_year=", work.get("publication_year"), "|", work.get("display_name"))
    fixed += int(ok)

authors_rows=[]
for (aid, yr), v in authors_map.items():
    inst_list = sorted(list(v["institutions"]), key=lambda x: (x[1] or "", x[0] or ""))
    authors_rows.append({
        "oa_author_id": v["oa_author_id"],
        "year": int(v["year"]),
        "author_name": v.get("author_name"),
        "orcid": v.get("orcid"),
        "institutions_json": json.dumps([{"id":iid,"name":iname} for iid,iname in inst_list], ensure_ascii=False),
        "raw_affiliation_strings_json": json.dumps(sorted(list(v["raw_affils"])), ensure_ascii=False),
        "n_institutions": len(inst_list),
    })
authors_df = pd.DataFrame(authors_rows)

print("fixed:", fixed)
print(papers_enriched_df.loc[papers_enriched_df["src_index"].isin([x[0] for x in FIXES]),
                            ["src_index","year","title","oa_status","oa_work_id","oa_doi_url"]])

papers_enriched_df["oa_status"].value_counts()

patched 7166 | OA_year= 2016 | Adaptive Accelerated Gradient Converging Methods under Holderian Error Bound Condition
patched 11690 | OA_year= 2021 | Skyformer: Remodel Self-Attention with Gaussian Kernel and Nyström Method
patched 21033 | OA_year= 2024 | Differential Privacy in Scalable General Kernel Learning via $K$-means Nystr{\"o}m Random Features
fixed: 3
       src_index  year  \
2715        7166  2017   
7239       11690  2021   
16582      21033  2024   

                                                                                                    title  \
2715            Adaptive Accelerated Gradient Converging Method under H\"{o}lderian Error Bound Condition   
7239                          Skyformer: Remodel Self-Attention with Gaussian Kernel and Nystr\"om Method   
16582  Differential Privacy in Scalable General Kernel Learning via $K$-means Nystr{\"o}m Random Features   

           oa_status                        oa_work_id  \
2715   ok_manual_doi  https://openal

,count
oa_status,
ok,20322
ok_manual_doi,3
error_http_400,1


In [16]:
# this confirms it doesnt exist on OpenAlex

import requests, json

doi_url = "https://doi.org/10.5555/3600270.3601909"

r = requests.get(
    "https://api.openalex.org/works",
    params={
        "filter": f"doi:{doi_url}",
        "per_page": 5,
        "select": "id,display_name,publication_year,doi,cited_by_count",
        "api_key": OPENALEX_API_KEY,
    },
    timeout=30,
)
print(r.status_code)
print(json.dumps(r.json(), indent=2)[:2000])

200
{
  "meta": {
    "count": 0,
    "db_response_time_ms": 12,
    "page": 1,
    "per_page": 5,
    "groups_count": null,
    "cost_usd": 0.0001
  },
  "results": [],
  "group_by": []
}


In [17]:
papers_enriched_df.loc[
    (papers_enriched_df["oa_status"].isin(["no_match","error_http_400"])) & (papers_enriched_df["year"]==2022),
    ["src_index","year","title","oa_status"]
]

,src_index,year,title,oa_status
11075,15526,2022,"Alleviating ""Posterior Collapse'' in Deep Topic Models via Policy Gradient",error_http_400


In [18]:
sid = 15526
m = papers_enriched_df["src_index"].astype(int) == sid

papers_enriched_df.loc[m, "oa_status"] = "missing_in_openalex"
papers_enriched_df.loc[m, "oa_http_status"] = 200
papers_enriched_df.loc[m, "oa_title_sim"] = None
papers_enriched_df.loc[m, "oa_work_id"] = None
papers_enriched_df.loc[m, "oa_doi_url"] = "https://doi.org/10.5555/3600270.3601909"
papers_enriched_df.loc[m, "oa_doi"] = "10.5555/3600270.3601909"
papers_enriched_df.loc[m, "oa_cited_by_count"] = None
papers_enriched_df.loc[m, "oa_primary_topic"] = None
papers_enriched_df.loc[m, "oa_topics_top5_json"] = None
papers_enriched_df.loc[m, "oa_author_ids_json"] = None
papers_enriched_df.loc[m, "oa_author_names_json"] = None
papers_enriched_df.loc[m, "oa_author_positions_json"] = None

papers_enriched_df.to_csv(PAPERS_PATH, index=False)
print(papers_enriched_df.loc[m, ["src_index","year","title","oa_status","oa_doi_url"]])

       src_index  year  \
11075      15526  2022   

                                                                            title  \
11075  Alleviating "Posterior Collapse'' in Deep Topic Models via Policy Gradient   

                 oa_status                               oa_doi_url  
11075  missing_in_openalex  https://doi.org/10.5555/3600270.3601909  


In [19]:
done = set(papers_enriched_df.loc[papers_enriched_df["oa_status"].isin(["ok","ok_fallback","ok_manual_doi","no_match","missing_in_openalex"]),
                                  "src_index"].dropna().astype(int).tolist())

In [21]:
papers_enriched_df['oa_status'].value_counts()

,count
oa_status,
ok,20322
ok_manual_doi,3
missing_in_openalex,1
